# Taller de programación (UBA) 
## Parte 2: Best Subset, Stepwise Selection & Métodos basados en la reducción de la dimensionalidad

**Objetivo:**  
Que se familiaricen con los métodos de selección de variables: Best Subset, Forward Stepwise y Backward Stepwise. Luego comparar con los métodos de regresión por componentes principales (PCR) y regresión de mínimos cuadrados parciales (PLS)

Vamos a seguir explotando la base `Hitters` (la misma que van a usar en el notebook de Regularización).

In [ ]:
import pandas as pd
import numpy as np
from ISLP import load_data
from sklearn.preprocessing import StandardScaler

from matplotlib import pyplot as plt
from matplotlib.pyplot import subplots

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from itertools import combinations

from functools import partial
import sklearn.model_selection as skm

import sklearn.linear_model as skl

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

### Datos: Hitters (baseballistas)

Datos de la Major League de Baseball en las temporadas 1986 y 1987. Nuestro objetivo es **predecir el salario** de los jugadores.

Es la misma base que van a usar en el notebook de Regularización (Ridge, LASSO, Elastic Net).

In [ ]:
Hitters = load_data('Hitters')
print('Dimensión original:', Hitters.shape)
print('Missings en Salary:', Hitters['Salary'].isnull().sum())

# Eliminamos filas sin salario
Hitters = Hitters.dropna().reset_index(drop=True)
print('Dimensión final:', Hitters.shape)

In [ ]:
Hitters.head()

### Preparar X e Y
Separamos la variable dependiente (Salary) y creamos dummies para las categóricas.

In [ ]:
y = Hitters['Salary']

# Creamos dummies para las variables categóricas
dummies = pd.get_dummies(Hitters[['League', 'Division', 'NewLeague']], drop_first=True)

# Armamos la matriz X con numéricas + dummies
X_num = Hitters.drop(['Salary', 'League', 'Division', 'NewLeague'], axis=1).astype('float64')
X_num = X_num.reset_index(drop=True) # index original es descartado

# Iniciamos el Standard Scaler
sc = StandardScaler()
# Estandarizamos las observaciones de entrenamiento
X_transformed = pd.DataFrame(sc.fit_transform(X_num),index=X_num.index, columns=X_num.columns)

X = pd.concat([X_transformed, dummies.astype('float64')], axis=1)

In [ ]:
# Chequeamos la matriz de predictores
print('Cantidad de predictores (p):', X.shape[1])
print('Cantidad de observaciones (n):', X.shape[0])
X.columns.tolist()

In [ ]:
X

### Train / Test split
Separamos 70% para entrenamiento y 30% para testeo (igual que en el notebook de CV).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=100)

print(f'Entrenamiento: {len(X_train)} observaciones')
print(f'Testeo: {len(X_test)} observaciones')

---
## 1. Best Subset Selection

La idea es probar **todas** las combinaciones posibles de variables y quedarse con el mejor modelo de cada tamaño.

Con $p = 19$ predictores, el total de modelos posibles es $2^{19} = 524.288$. Es factible pero costoso.

Para simplificar, vamos a hacer Best Subset hasta un máximo de variables (por ejemplo, hasta 10).

In [ ]:
def best_subset(X_train, y_train, max_vars=10):
    '''
    Para cada tamaño k (de 1 a max_vars), encuentra el mejor subconjunto 
    de k variables (el que tiene menor RSS en la muestra de entrenamiento).
    
    Devuelve un diccionario con el mejor modelo de cada tamaño.
    '''
    resultados = {}
    todas_las_vars = X_train.columns.tolist()
    
    for k in range(1, max_vars + 1):
        mejor_rss = np.inf
        mejor_vars = None
        
        # Probar todas las combinaciones de k variables
        for combo in combinations(todas_las_vars, k):
            modelo = LinearRegression().fit(X_train[list(combo)], y_train)
            rss = np.sum((y_train - modelo.predict(X_train[list(combo)]))**2)
            
            if rss < mejor_rss:
                mejor_rss = rss
                mejor_vars = list(combo)
        
        resultados[k] = {'variables': mejor_vars, 'rss_train': mejor_rss}
        print(f'k={k}: {len(list(combinations(todas_las_vars, k)))} modelos evaluados -> mejores vars: {mejor_vars}')
    
    return resultados

In [ ]:
# Esto tarda un poco porque prueba muchas combinaciones
# Con max_vars=4 es rápido; con max_vars=10 tarda ~1 min
best_models = best_subset(X_train, y_train, max_vars=4)

#### Elegir el mejor tamaño con MSE de testeo
Ahora evaluamos cada modelo en la **muestra de testeo** para elegir cuántas variables incluir.

In [ ]:
mse_best = []

for k, info in best_models.items():
    vars_k = info['variables']
    modelo = LinearRegression().fit(X_train[vars_k], y_train)
    y_pred = modelo.predict(X_test[vars_k])
    mse = mean_squared_error(y_test, y_pred)
    mse_best.append({'k': k, 'mse_test': mse, 'variables': vars_k})

df_best = pd.DataFrame(mse_best)
df_best

In [ ]:
# Mejor modelo según MSE de testeo
mejor = df_best.loc[df_best['mse_test'].idxmin()]
print(f"Mejor modelo: k={int(mejor['k'])} variables")
print(f"MSE test: {mejor['mse_test']:.2f}")
print(f"Variables: {mejor['variables']}")

In [ ]:
# Gráfico: MSE test vs número de variables
plt.figure(figsize=(8, 5))
plt.plot(df_best['k'], df_best['mse_test'], 'o-', color='steelblue')
plt.xlabel('Número de variables (k)')
plt.ylabel('MSE de testeo')
plt.title('Best Subset Selection: MSE test según tamaño del modelo')
plt.xticks(df_best['k'])
plt.grid(False)
plt.show()

#### Limitación de Best Subset

Con $p = 19$ y `max_vars=4`, ya estamos evaluando miles de modelos. Si quisiéramos ir hasta $k=19$, serían $2^{19} \approx 524.000$ modelos.

**Pregunta para pensar:** ¿Qué pasa si tenemos $p = 100$ o $p = 284$ variables (como en Belloni et al., 2014)?

---
## 2. Forward Stepwise Selection

En lugar de probar todas las combinaciones, **empezamos de cero** y vamos agregando variables de a una: en cada paso, agregamos la que más reduce el RSS.

In [ ]:
def forward_stepwise(X_train, y_train):
    '''
    Forward Stepwise Selection.
    Empieza sin variables y agrega de a una la que más reduce el RSS.
    '''
    todas_las_vars = X_train.columns.tolist()
    incluidas = []
    resultados = {}
    
    for k in range(1, len(todas_las_vars) + 1):
        mejor_rss = np.inf
        mejor_var = None
        
        # Probar agregar cada variable que aún no está incluida
        for var in todas_las_vars:
            if var not in incluidas:
                candidatas = incluidas + [var]
                modelo = LinearRegression().fit(X_train[candidatas], y_train)
                rss = np.sum((y_train - modelo.predict(X_train[candidatas]))**2)
                
                if rss < mejor_rss:
                    mejor_rss = rss
                    mejor_var = var
        
        incluidas.append(mejor_var)
        resultados[k] = {'variables': incluidas.copy(), 'rss_train': mejor_rss}
    
    return resultados

In [ ]:
fwd_models = forward_stepwise(X_train, y_train)

# Veamos el orden en que se agregan las variables
print('Orden de entrada de variables (Forward):')
for k in range(1, len(fwd_models) + 1):
    nueva = fwd_models[k]['variables'][-1]  # última variable agregada
    print(f'  Paso {k}: + {nueva}')

---
## 3. Backward Stepwise Selection

Lo opuesto: **empezamos con todas** las variables y vamos sacando de a una la que menos aporta.

In [ ]:
def backward_stepwise(X_train, y_train):
    '''
    Backward Stepwise Selection.
    Empieza con todas las variables y saca de a una la que menos aporta.
    '''
    incluidas = X_train.columns.tolist()
    resultados = {}
    p = len(incluidas)
    
    # Modelo completo
    modelo_full = LinearRegression().fit(X_train[incluidas], y_train)
    rss_full = np.sum((y_train - modelo_full.predict(X_train[incluidas]))**2)
    resultados[p] = {'variables': incluidas.copy(), 'rss_train': rss_full}
    
    for k in range(p, 1, -1):
        mejor_rss = np.inf
        peor_var = None
        
        # Probar sacar cada variable que está incluida
        for var in incluidas:
            candidatas = [v for v in incluidas if v != var]
            modelo = LinearRegression().fit(X_train[candidatas], y_train)
            rss = np.sum((y_train - modelo.predict(X_train[candidatas]))**2)
            
            if rss < mejor_rss:
                mejor_rss = rss
                peor_var = var
        
        incluidas.remove(peor_var)
        resultados[k - 1] = {'variables': incluidas.copy(), 'rss_train': mejor_rss}
    
    return resultados

In [ ]:
bwd_models = backward_stepwise(X_train, y_train)

# Veamos el orden en que se sacan las variables
print('Orden de salida de variables (Backward):')
p = len(X_train.columns)
for k in range(p, 1, -1):
    vars_antes = set(bwd_models[k]['variables'])
    vars_despues = set(bwd_models[k-1]['variables'])
    sacada = vars_antes - vars_despues
    print(f'  Paso {p - k + 1}: - {sacada.pop()}')

---
## 4. Comparación de los tres métodos

Evaluamos el MSE de testeo para cada tamaño de modelo en los tres métodos.

In [ ]:
def evaluar_en_test(modelos, X_train, X_test, y_train, y_test):
    '''Calcula el MSE de testeo para cada modelo de la secuencia.'''
    resultados = []
    for k, info in sorted(modelos.items()):
        vars_k = info['variables']
        modelo = LinearRegression().fit(X_train[vars_k], y_train)
        y_pred = modelo.predict(X_test[vars_k])
        mse = mean_squared_error(y_test, y_pred)
        resultados.append({'k': k, 'mse_test': mse})
    return pd.DataFrame(resultados)

In [ ]:
mse_fwd = evaluar_en_test(fwd_models, X_train, X_test, y_train, y_test)
mse_bwd = evaluar_en_test(bwd_models, X_train, X_test, y_train, y_test)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mse_fwd['k'], mse_fwd['mse_test'], 'o-', label='Forward', alpha=0.8)
plt.plot(mse_bwd['k'], mse_bwd['mse_test'], 's-', label='Backward', alpha=0.8)

# Agregar Best Subset si lo corrimos
if len(df_best) > 0:
    plt.plot(df_best['k'], df_best['mse_test'], 'D-', label='Best Subset', alpha=0.8)

plt.xlabel('Número de variables (k)')
plt.ylabel('MSE de testeo')
plt.title('Comparación: Best Subset vs Forward vs Backward')
plt.legend()
plt.grid(False)
plt.show()

In [ ]:
# Mejor modelo de cada método
mejor_fwd = mse_fwd.loc[mse_fwd['mse_test'].idxmin()]
mejor_bwd = mse_bwd.loc[mse_bwd['mse_test'].idxmin()]

print(f"Forward:  k={int(mejor_fwd['k'])}, MSE test={mejor_fwd['mse_test']:.2f}")
print(f"Backward: k={int(mejor_bwd['k'])}, MSE test={mejor_bwd['mse_test']:.2f}")

print(f"\nVariables del mejor modelo Forward:")
print(fwd_models[int(mejor_fwd['k'])]['variables'])

print(f"\nVariables del mejor modelo Backward:")
print(bwd_models[int(mejor_bwd['k'])]['variables'])

#### ¿Forward y Backward eligen las mismas variables?

No necesariamente. Como son algoritmos *greedy* (toman la mejor decisión en cada paso sin volver atrás), pueden llegar a soluciones distintas o iguales

In [ ]:
# Comparamos las variables elegidas por Forward y Backward para k=5
k_comparar = 9
vars_fwd = set(fwd_models[k_comparar]['variables'])
vars_bwd = set(bwd_models[k_comparar]['variables'])

print(f'Variables en Forward (k={k_comparar}):  {vars_fwd}')
print(f'Variables en Backward (k={k_comparar}): {vars_bwd}')
print(f'\nEn común: {vars_fwd & vars_bwd}')
print(f'Solo en Forward: {vars_fwd - vars_bwd}')
print(f'Solo en Backward: {vars_bwd - vars_fwd}')

---
## 5. Resumen y conexión con lo que vimos en Regularización

| Método | Ventaja | Limitación |
|--------|---------|------------|
| **Best Subset** | Encuentra el óptimo global | Inviable si $p$ es grande ($2^p$ modelos) |
| **Forward** | Rápido, funciona con $p > n$ | Greedy: no garantiza el óptimo |
| **Backward** | Rápido | Requiere $n > p$ |

### ¿Qué vimos en Regularización?

**LASSO** resuelve el mismo problema (seleccionar variables) pero de forma continua y más eficiente. En vez de incluir/excluir variables, achica los coeficientes y algunos los lleva exactamente a cero.


##### Tarea para la casa:
- Usar **K-Fold CV** (en vez de un solo train/test split) para elegir el número óptimo de variables en Forward Stepwise. ¿Cambia el resultado?
- Comparar el MSE del mejor modelo de Forward con el MSE del modelo de LASSO del notebook de Regularización.

---
## 6. Métodos basados en Reducción de la dimensionalidad

### 6.1. Regresión por Componentes Principales (PCR, principal component regression)
Para estimar por una regresión por componentes principales (PCR), primero, usamos `PCA` del modulo de [PCA() de Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html). 
Luego, usamos la funcion conocida de `LinearRegression()` para estimar el modelo.

Utilizamos la función [Pipeline()](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) de Sckit learn que permite en una forma clara, separar el paso de *transformar* los predictores y luego *estimar* el modelo de interés. 

In [ ]:
pca_2 = PCA(n_components=2)
linreg_2 = skl.LinearRegression()
pipe = Pipeline([('pca', pca_2),
                 ('linreg', linreg_2)])
pipe.fit(X_train, y_train)
print(f'Los coeficientes de la regresión usando dos componentes principales son:')
print(pipe.named_steps['linreg'].coef_.round(2))


Podemos acceder a los ponderadores de PCA igual que antes (formalmente los $\phi_{jm}$ en clase)

In [ ]:
pipe.named_steps['pca'].components_.round(3)

Con el atributo `explained_variance_ratio_` de `PCA` podemos ver el porcentaje de la varianza explicada en los predictores y en la respuesta usando distinto número de componentes principales. 

In [ ]:
pipe.named_steps['pca'].explained_variance_ratio_.round(2)

Esto nos dice la cantidad de información sobre los predictores que se captura usando $M$ componentes. Esto nos muestra que usar $M=1$ capta un 38% de la varianza, mientras que $M=2$ capta un 22%.

In [ ]:
# Evaluación en TEST SET
y_pred_pcr_2_test = pipe.predict(X_test)
mse_pcr_2_test = mean_squared_error(y_test, y_pred_pcr_2_test)
r2_pcr_2_test = r2_score(y_test, y_pred_pcr_2_test)

print(f"MSE test: {mse_pcr_2_test:.2f}")
print(f"R² test:  {r2_pcr_2_test:.4f}")

# Varianza explicada
var_explicada = pipe.named_steps['pca'].explained_variance_ratio_.sum()
print(f"Varianza explicada con 2 componentes como predictores: {var_explicada*100:.1f}%")

#### 6.1.1. Cross-validation para elegir el número de componentes en PCR 

Podemos usar cross-validation (CV) para buscar el número de componentes a utilizar en la regresión. Para eso, usaremos `skm.GridSearchCV`, donde el parámetro que varía es `n_components`.

In [ ]:
# Usamos 5-fold cross-validation
K = 5
kfold = skm.KFold(K,
                  random_state=0,
                  shuffle=True)

# Definimos un rango de numero de componentes principales
param_grid = {'pca__n_components': range(1, 20)}


grid = skm.GridSearchCV(pipe,
                        param_grid,
                        cv=kfold,
                        scoring='neg_mean_squared_error')
#Estimamos
grid.fit(X_train, y_train)

Graficamos los errores de cross-validation y elegir el número optimo de componentes a usar como regresores

In [ ]:
# Figura de Error de Cross validation
pcr_fig, ax = subplots(figsize=(10, 6))
n_comp = list(param_grid['pca__n_components'])
mean_mse = -grid.cv_results_['mean_test_score']
std_mse = grid.cv_results_['std_test_score'] / np.sqrt(K)

ax.errorbar(n_comp, mean_mse, std_mse, fmt='o-', 
            color='steelblue', capsize=5, capthick=2, alpha=0.8)
ax.set_ylabel('CV MSE', fontsize=12)
ax.set_xlabel('# componentes principales', fontsize=12)
ax.set_title('PCR: Cross-Validation MSE vs Número de Componentes', fontsize=13)
ax.set_xticks(n_comp[::2])
plt.tight_layout()
plt.show()

#### 6.1.2. Validación con el número óptimo de componentes elegido por CV

In [ ]:
# Obtener número óptimo de componentes
best_n_comp_pcr = grid.best_params_['pca__n_components']
best_mse_cv_pcr = -grid.best_score_

print(f"Número óptimo de componentes (PCR): {best_n_comp_pcr}")
print(f"MSE promedio en CV (5-fold): {best_mse_cv_pcr:.2f}")

In [ ]:
# Entrenar modelo final con número óptimo en X_train completo
pca_final = PCA(n_components=best_n_comp_pcr)
linreg_final = skl.LinearRegression()
pipe_final = Pipeline([('pca', pca_final),
                       ('linreg', linreg_final)])
pipe_final.fit(X_train, y_train)

# Evaluación en TEST SET
y_pred_pcr_test = pipe_final.predict(X_test)
mse_pcr_test = mean_squared_error(y_test, y_pred_pcr_test)
r2_pcr_test = r2_score(y_test, y_pred_pcr_test)

print(f"MSE test: {mse_pcr_test:.2f}")

In [ ]:
# Varianza explicada
var_explicada = pipe_final.named_steps['pca'].explained_variance_ratio_.sum()
print(f"Varianza explicada en predictores: {var_explicada*100:.5f}%")

**Pregunta**: Hubo reducción de la dimensionalidad con PCR?

### 6.2. Mínimos Cuadrados Parciales (Partial Least Squares, PLS)
Minimos Cuadrados Parciales (PLS) se implementa con la funcion [PLSRegression()](https://scikit-learn.org/stable/modules/generated/sklearn.cross_decomposition.PLSRegression.html)


In [ ]:
pls = PLSRegression(n_components=2, 
                    scale=True)
pls.fit(X_train, y_train)

Veamos que obtenemos cuando usamos el atributo `coef_` luego de estimar por PLS

In [ ]:
# Coeficientes con nombres de variables
coef_pls = pd.DataFrame({
    'Variable': X_train.columns,
    'Coeficiente': pls.coef_.flatten()
}).sort_values('Coeficiente', key=abs, ascending=False)

print(coef_pls)

En PLS, los coeficientes que vemos son los **pesos** de la regresión lineal final (después de la transformación a componentes). A diferencia de PCR, PLS considera tanto X como y al construir los componentes, por eso sus coeficientes pueden diferir.

Si además quieres ver los componentes PLS (cómo se construyeron):

In [ ]:
# Componentes PLS
print("Componentes PLS:")
print(pls.x_weights_)  # pesos de X

##### Diferencia conceptual:
`pls.x_weights_` (forma: `n_components × n_variables`): Son los **pesos/direcciones** que definen cómo se construyen los componentes PLS. Cada fila es una dirección en el espacio de X que maximiza la covarianza con Y.

`pls.coef_` (forma: `n_variables`): Son los **coeficientes finales de regresión** de Y en función de X original (no de los componentes). Es decir, si predices con pls.predict(X), internamente hace:

1. Transforma X → Z (componentes) usando x_weights_
2. Regresiona Y ~ Z
3. **Proyecta los coeficientes de vuelta a escala de X original** → eso es coef_

**No son betas de regresiones auxiliares**

Los `coef_` **NO** son los coeficientes de las regresiones auxiliares univariadas de Y en cada X_j. ISLP los describe como:

Los componentes Z se construyen iterativamente buscando direcciones que maximicen Cov(X, Y), y luego se regresiona Y ~ Z.


In [ ]:
# Comparar x_weights_ vs coef_
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# x_weights (componente 1)
axes[0].barh(X_train.columns, pls.x_weights_[:, 0])
axes[0].set_title(f'PLS x_weights_: Componente 1\n(Direcciones que definen Z1)', fontsize=12)
axes[0].set_xlabel('Peso')

# coef_ (coeficientes finales)
axes[1].barh(X_train.columns, pls.coef_.flatten())
axes[1].set_title('PLS coef_: Coeficientes finales\n(Después de proyectar Z → X)', fontsize=12)
axes[1].set_xlabel('Coeficiente')

plt.tight_layout()
plt.show()

#### 6.2.1. Validación con el número óptimo de componentes elegido por CV
Similar al caso de PCR, usamos CV para elegir el numero de componentes

In [ ]:
param_grid = {'n_components':range(1, 20)}

grid = skm.GridSearchCV(pls,
                        param_grid,
                        cv=kfold,
                        scoring='neg_mean_squared_error')

grid.fit(X_train, y_train)

De la misma forma, en un graficamos el MSE por CV

In [ ]:
# Figura mejorada: consistente con sección 4
pls_fig, ax = subplots(figsize=(10, 6))
n_comp = list(param_grid['n_components'])
mean_mse = -grid.cv_results_['mean_test_score']
std_mse = grid.cv_results_['std_test_score'] / np.sqrt(K)

ax.errorbar(n_comp, mean_mse, std_mse, fmt='s-', 
            color='coral', capsize=5, capthick=2, alpha=0.8)
ax.set_ylabel('CV MSE', fontsize=12)
ax.set_xlabel('# componentes', fontsize=12)
ax.set_title('PLS: Cross-Validation MSE vs Número de Componentes', fontsize=13)
ax.set_xticks(n_comp[::2])
ax.grid(False)
plt.tight_layout()
plt.show()

#### 6.2.2. Validación con el número óptimo de componentes elegido por CV

In [ ]:
# Obtener número óptimo de componentes
best_n_comp_pls = grid.best_params_['n_components']
best_mse_cv_pls = -grid.best_score_

print(f"Número óptimo de componentes (PLS): {best_n_comp_pls}")
print(f"MSE promedio en CV (5-fold): {best_mse_cv_pls:.2f}")

# Entrenar modelo final con número óptimo
pls_final = PLSRegression(n_components=best_n_comp_pls, scale=True)
pls_final.fit(X_train, y_train)

In [ ]:
# Evaluación en TEST SET
y_pred_pls_test = pls_final.predict(X_test)
mse_pls_test = mean_squared_error(y_test, y_pred_pls_test)
r2_pls_test = r2_score(y_test, y_pred_pls_test)

print(f"MSE test: {mse_pls_test:.2f}")
print(f"R² test:  {r2_pls_test:.4f}")

### 6.3. Comparación: PCR vs PLS

In [ ]:
# Comparar ambos métodos
comparacion = pd.DataFrame({
    'Método': ['PCR', 'PLS'],
    'Componentes óptimos': [best_n_comp_pcr, best_n_comp_pls],
    'MSE CV': [best_mse_cv_pcr, best_mse_cv_pls],
    'MSE Test': [mse_pcr_test, mse_pls_test],
    'R² Test': [r2_pcr_test, r2_pls_test]
})

comparacion.round(2)



Si vamos a la Parte 1 de esta clase de regularización, podemos comparar la performance de todos los métodos para este caso de predicción de salarios de baseballistas: 

In [ ]:
# dataframe con modelos previos (Lineal, Ridge, LASSO)
df_compara_regularizacion = pd.DataFrame({
    'Modelo': ['Lineal_OLS', 'Ridge_alpha_cv', 'LASSO_alpha_cv', 'PCR', 'PLS'],
    'MSE_test': [123843.30, 115974.76, 112091.67, mse_pcr_test, mse_pls_test]
})

df_compara_regularizacion = df_compara_regularizacion.sort_values('MSE_test')

df_compara_regularizacion.round(2)

**Tarea para la casa**

Grafico que compare coeficientes de
- Lineal_OLS
- Foward stepselection (con K por cv)
- Backward stepselection